In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
!pip install langchain langchain-openai langchain-community langgraph python-dotenv langchain-mcp-adapters langchain-chroma chromadb pypdf

In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model('openai:gpt-5-mini');

# Tools

Tools are just methods with proper defined input, output and description

In [ ]:
from pydantic import BaseModel
class MovieShows(BaseModel):
  name: str
  timing: str

response = model.with_structured_output(MovieShows).invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [ ]:
print(response)

In [ ]:
from langchain_core.tools import tool

In [ ]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

Tool -> Args with type hints.

Tools are just glorified Functions/API Calls

PS: If we don't know how to write good function we won't be able to make good tools.

In [ ]:
@tool('book_seats', description = 'Book Cinema for a customer, use whenever customer wants to book/reserve a seat.')
def reserve(movie:str,seats:int) ->str:
  """Reserve Seats"""
  return f"Reserved {seats} seat for {movie}"

@tool is a langchain supported decorator

Trying tools provided by langchain docs

Ex: Calling Tavily which is used for web search or research

In [ ]:
!pip install -qU langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch

@tool('search_internet_with_tavily', description = 'Use this when user wants to search the internet with Tavily')
def search_internet(topic):
  return TavilySearch()

tool = TavilySearch(
    max_results=5,
    topic="general"
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

## args_schema



In [ ]:
from pydantic import Field
from typing import Literal

class SeatBookingInput(BaseModel):
    movie_title:str = Field(description='Exact Movie Title')
    seat_count : int = Field(description='Number of seats to book', ge=1, le=10)
    preferred_row : Literal['front', 'middle', 'back'] = Field(default='middle', description='Preferred seat row')

Binding our function in @tool gives our agent a lot better understanding which helps agent to call our functions as tools when needed.

If we don't use @tool decorater still we can use function but then chances of error will increase a lot

In [ ]:
@tool(args_schema=SeatBookingInput)
def book_seats(movie_title:str, seats:int, preferred_row:str)-> str:
  """Book Seats for a Movie"""
  return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

In [ ]:
print(book_seats.args)

In [ ]:
# {
#   'movie_title': {
#     'description': 'Exact Movie Title',
#     'title': 'Movie Title', 'type': 'string'
#   },
#   'seat_count': {'description': 'Number of seats to book', 'maximum': 10, 'minimum': 1, 'title': 'Seat Count', 'type': 'integer'},
#   'preferred_row': {'default': 'middle', 'description': 'Preferred seat row', 'enum': ['front', 'middle', 'back'], 'title': 'Preferred Row', 'type': 'string'}
# }

Let's handle config etc. maybe let's say your have theatre

### Never use _config_ and _runtime_ as args or parameter of Tool

These are reserved keywords

In [19]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool
def get_weather(location: str,config:str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} {config} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model='openai:gpt-5-mini',
    tools=[get_weather]
    )


running below throws error because config is reserved keyword

In [ ]:
result = agent.invoke(
        {"messages": [{"role": "user", "content": "What is the weather in Delhi in celsius and tell the forecast?"}]},

)

## Binding vs Execution

In [ ]:
model

In [ ]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema."""
  return "Show is available"

In [ ]:
model_with_tools = model.bind_tools([check_showtimes, book_seats])

In [ ]:
response = model_with_tools.invoke("Is Interstellar showing tonight, and can you book me 2 seats?")

In [ ]:
response

## Runtime in Tools

A runtime param in tool which our tool can use to read a lot of things in code and otherwise as well

In [14]:
from langchain.tools import ToolRuntime
from langchain_core.messages import HumanMessage

@tool
def get_last_movie_mentioned(movie: str, runtime: ToolRuntime) -> str:
  """Get the last movie mentioned in the chat history."""
  pass

print(get_last_movie_mentioned.args)

{'movie': {'title': 'Movie', 'type': 'string'}}


In [15]:
!pip install langgraph

InMemorySave() is a checkpoint storage class in Langgraph ecosystem (part of Langchain/ LangGraph family). It stores the graph's state only in RAM (memory)

In [22]:
from langchain_core.tools import tool
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

loyalty_store= InMemoryStore()


@tool
def save_favourite_genres(customer_id:str,genre:str,runtime:ToolRuntime) -> str:
  """Save a customer's facvourite movie genre for future visits"""
  runtime.store.put((customer_id,"preferences"),"favourite_genre",{"value":genre})
  return f"Got it -- I will remmeber you like {genre} movies"

@tool
def recall_favourite_genre(customer_id:str,runtime:ToolRuntime) -> str:
  """ Recall a customer's fav movie genre, if we have saved it before"""
  favourite_genre = runtime.store.get((customer_id,"preferences"),"favourite_genre")
  return favourite_genre.value["value"] if favourite_genre else "We don't have any saved preference for this user"


memory_agent = create_agent(
    model = model,
    tools=[save_favourite_genres,recall_favourite_genre],
    store=loyalty_store  # Attached to the agent, tools can access it using runtime)
)

Here's how you can use the memory_agent to save and recall favorite genres, which utilizes the InMemoryStore we set up

In [25]:
# Save a favorite genre using the memory_agent
save_result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My favorite movie genre is Sci-Fi, and my customer ID is Jatin123 please remember that"
    }]
})

print(save_result)
print(f"Save Result: {save_result['messages'][-1].content}")

{'messages': [HumanMessage(content='My favorite movie genre is Sci-Fi, and my customer ID is Jatin123', additional_kwargs={}, response_metadata={}, id='6230bd9c-820f-4b55-8281-8574e4f22410'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 98, 'prompt_tokens': 187, 'total_tokens': 285, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E6eomS59kJ8IvUWFRfOez6IAKaugF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fa986-5249-74e0-bad0-04c8e7eafed3-0', tool_calls=[{'name': 'save_favourite_genres', 'args': {'customer_id': 'Jatin123', 'genre': 'Sci-Fi'}, 'id': 'call_CC2nso9UymGjhIjsLGfrnOLG

In [24]:
# Recall the favorite genre for the same customer
recall_result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is Jatin123's favorite movie genre?"
    }]
})
print(f"Recall Result: {recall_result['messages'][-1].content}")

Recall Result: Jatin123's favorite movie genre is Sci-Fi.


In this example:

- The `loyalty_store` (an `InMemoryStore`) is passed to the `create_agent` function, allowing tools to access it via `runtime.store`.
- When you invoke the `memory_agent` with a message that triggers `save_favourite_genres`, the genre is stored in `loyalty_store` using the `put` method.
- When you invoke it with a message that triggers `recall_favourite_genre`, the agent retrieves the stored genre using the `get` method.

This demonstrates how `InMemoryStore` acts as a temporary, in-memory database for your agent's tools to share and persist state within a single session.

tool is just a node in langgraph
### Two key learnings:



- ```runtime.execution_info```: details about execution of runtime like run_id, node_attempt and all
- ```runtime.server_info```: valid on Langchain server and is None for local development

In [ ]:
@tool
def log_booking_context(runtime: ToolRuntime) -> str:
  info = runtime.execution_info

  # run_id
  # node_attempt


### Skipping the Model's final Polishing

In [26]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Tickets are refundable up to 2 hours before showtime. No refunds after that."

direct_agent = create_agent(model="openai:gpt-5-mini", tools=[get_exact_refund_policy])
result = direct_agent.invoke({"messages": [("user", "What's your refund policy? Please explain in points")]})
print(result["messages"][-1].content)

Tickets are refundable up to 2 hours before showtime. No refunds after that.


## Dynamic Tool Loading & Calling

In [28]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@tool
def standard_booking(movie_title: str) -> str:
  """Book a standard seat"""
  return f"Standard seat booked for {movie_title}."

@tool
def vip_lounge_booking(movie_title: str) -> str:
  """Book a VIP Lounge with premium service. VIP members only"""
  return f"VIP seat booked for {movie_title}."

gated_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[standard_booking, vip_lounge_booking]
)

In [29]:
result_regular = gated_agent.invoke({"messages": [("user", "Book me a VIP lounge seat for Dune?")]})


In [30]:
result_regular

{'messages': [HumanMessage(content='Book me a VIP lounge seat for Dune?', additional_kwargs={}, response_metadata={}, id='00f40fbe-6968-4ae6-97af-5aedc4f49599'),
  AIMessage(content='I can do that — VIP lounge bookings are for VIP members only. Before I book, I need a few details:\n\n1. Which "Dune" do you mean (Dune 2021, Dune: Part Two 2024, or another release)?  \n2. City / theater name (or allow me to find nearby theaters for you).  \n3. Date and showtime you want (or a range, e.g., "this weekend evening").  \n4. How many VIP seats?  \n5. Are you a VIP member of the theater chain? If not, do you want help enrolling or would you like me to check availability anyway (some locations let non-members purchase VIP access at extra cost)?  \n6. Contact/email and payment method preference (so I know how to complete the booking) — if you prefer, I can just reserve and tell you how to finish payment.\n\nTell me the answers you have and I’ll proceed.', additional_kwargs={'refusal': None}, resp